# AIM 402 - My Course Dataset (simulated)

Simulated dataset: **Nepal Student Performance** (1,200 students).
Generated with NumPy using a fixed seed, so the same data is created every time.
Some values are missing on purpose, to make it behave like real survey data.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(402)   # fixed seed = same dataset every run
n = 1200

# ---------- NUMERIC VARIABLES ----------
age = rng.integers(17, 25, n)                                        # 17-24
study_hours = np.clip(rng.gamma(2.5, 1.6, n), 0, 14)                 # per day, right-skewed
sleep_hours = np.clip(rng.normal(6.8, 1.2, n), 3, 10)                # per night
attendance_pct = np.clip(rng.normal(82, 11, n), 30, 100)             # %
family_income = np.round(rng.lognormal(3.6, 0.6, n), 1)              # NPR thousand / month, skewed

# ---------- CATEGORICAL VARIABLES ----------
location_type = rng.choice(["Urban", "Semi-urban", "Rural"], n, p=[0.45, 0.30, 0.25])        # 3 levels
stream = rng.choice(["Science", "Management", "Humanities", "Education"], n,
                    p=[0.30, 0.35, 0.20, 0.15])                                              # 4 levels

# binary: tuition more likely when income is higher
p_tuition = 1 / (1 + np.exp(-(np.log(family_income) - 3.6) * 1.5))
has_tuition = np.where(rng.random(n) < p_tuition, "Yes", "No")                               # binary

# ---------- EXAM SCORE (depends on the other variables + noise) ----------
loc_effect = np.select([location_type == "Urban", location_type == "Semi-urban"], [3, 0], default=-3)
stream_effect = np.select([stream == "Science", stream == "Management", stream == "Humanities"],
                          [2, 1, -1], default=0)
noise = rng.normal(0, 7, n)

exam_score = (12 + 4.0 * study_hours + 0.25 * attendance_pct + 1.5 * (sleep_hours - 6.8)
              + 6 * (has_tuition == "Yes") + loc_effect + stream_effect + noise)
exam_score = np.round(np.clip(exam_score, 0, 100), 1)

result = np.where(exam_score >= 40, "Pass", "Fail")                                          # binary

# ---------- PUT IN A TABLE ----------
df = pd.DataFrame({
    "student_id": np.arange(1, n + 1),
    "age": age,
    "study_hours": np.round(study_hours, 1),
    "sleep_hours": np.round(sleep_hours, 1),
    "attendance_pct": np.round(attendance_pct, 1),
    "family_income": family_income,
    "location_type": location_type,
    "stream": stream,
    "has_tuition": has_tuition,
    "exam_score": exam_score,
    "result": result,
})

# ---------- ADD MISSING VALUES (random blanks) ----------
missing_rate = {"study_hours": 0.03, "sleep_hours": 0.05, "attendance_pct": 0.04,
                "family_income": 0.12, "location_type": 0.02, "has_tuition": 0.03}

for col, rate in missing_rate.items():
    mask = rng.random(n) < rate
    df[col] = df[col].where(~mask, np.nan)

# a few unusual (outlier) values, like real data entry mistakes
bad = rng.choice(n, 6, replace=False)
df.loc[bad[:3], "study_hours"] = 16.0      # unrealistic study hours
df.loc[bad[3:], "family_income"] = 950.0   # very high income

df.to_csv("nepal_student_performance.csv", index=False)
print("Saved:", df.shape)
df.head()

## Quick checks

In [ ]:
print(df.shape)
print()
print("Missing values per column:")
print(df.isna().sum())
print()
df.describe()

In [ ]:
# Does it meet the course requirements?
num_cols = df.select_dtypes("number").columns.drop("student_id").tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()

print("Rows >= 500        :", len(df) >= 500)
print("Numeric columns    :", num_cols)
print("Categorical columns:", {c: df[c].nunique() for c in cat_cols})
print("Has a binary column       :", any(df[c].nunique() == 2 for c in cat_cols))
print("Has a 3+ level column     :", any(df[c].nunique() >= 3 for c in cat_cols))